# MedCourse — Google Colab Ultra 备用运行环境

> **合规与适用边界**：本 Notebook 仅用于**完全不涉及真实患者/医院数据**的任务——包括全仓单元测试、**路线 B（基于公开卫健委临床路径与抗菌药指南的知识建表）**以及**基于内置虚构统计夹具的合成流水线验证**。
> **推荐硬件**：Colab Ultra 请选择 **`A100 GPU`（40GB/80GB）** 或 **`L4 GPU`（24GB）** + **`High-RAM`**；如需长时间扩展运行，可启用后台执行并挂载 Google Drive 持久化模型缓存与输出目录。

In [ ]:
# 1. 检查挂载显卡 & 可选挂载 Google Drive 持久化缓存权重和输出
!nvidia-smi

USE_DRIVE = True  # 若无需挂载 Google Drive 可改为 False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_ROOT = "/content/drive/MyDrive/MedCourse_Ultra_Cache"
else:
    CACHE_ROOT = "/content/medcourse_cache"

import os
HF_CACHE = os.path.join(CACHE_ROOT, "hf_models")
OUT_ROOT = os.path.join(CACHE_ROOT, "outputs")
os.makedirs(HF_CACHE, exist_ok=True)
os.makedirs(OUT_ROOT, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
print("HF_CACHE:", HF_CACHE)
print("OUT_ROOT:", OUT_ROOT)

In [ ]:
# 2. 拉取仓库并运行全套 MANIFEST 校验 + 单元测试（含 bench_core 端到端）
import os, subprocess

REPO_DIR = "/content/MedCourse"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Panyto/MedCourse.git /content/MedCourse
else:
    !git -C /content/MedCourse pull --ff-only

%cd /content/MedCourse
!python scripts/run_all_tests.py

In [ ]:
# 3. 安装 vLLM 推理服务（若重连实例需重新执行此步；模型权重会优先复用 Drive 缓存）
!pip install -q --upgrade vllm
!python -c "import vllm; print('vLLM version:', vllm.__version__)"

In [ ]:
# 4. 启动仅绑定 127.0.0.1 的本地 vLLM 服务（兼容 medcourse.llm.Client；v0.3.2 合成推荐 Qwen/Qwen3-32B-AWQ，路线 B 抽取可用 Qwen/Qwen3-8B）
from colab import colab_bootstrap as cb

cb.setup_env("/content/MedCourse")
cb.inspect_gpu()

MODEL_ID = "Qwen/Qwen3-32B-AWQ"  # 仅跑路线 B 抽取可切换为 "Qwen/Qwen3-8B"（max_model_len=8192）
SERVED_NAME = MODEL_ID.split("/")[-1]
PORT = 8101

srv = cb.start_vllm(
    model=MODEL_ID,
    port=PORT,
    served_name=SERVED_NAME,
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    dtype="auto",
    download_dir=HF_CACHE,
)

In [ ]:
# 5. 主力备用场景 A：运行公开临床路径「路线 B (route_b_kit v0.5)」多轮检索建表
# 默认先试跑 2 个代表病种（肺炎 + 急性胆囊炎）；设 ONLY = "" 可一次跑完全部 17 病种
import pandas as pd

ONLY = "J15,K80.0_acute"
rb_out = cb.run_route_b(
    out_dir=f"{OUT_ROOT}/route_b_colab_run",
    only=ONLY,
    concurrency=6,
    rounds=3,
    k=3,
    extra=3,
    port=PORT,
    model=SERVED_NAME,
)
display(pd.read_csv(rb_out / "grounded_by_field.csv"))

In [ ]:
# 6. 备用场景 B：100% 公开路线 B (--kb-only, kb_v22.json) 国标 7 类文书生成演示 + 释放服务
import json

summary = cb.run_kb_only_demo(
    out_dir=f"{OUT_ROOT}/kb_only_colab_demo",
    n=2,
    use_llm=True,
    port=PORT,
    model=SERVED_NAME,
)
with open(summary["out_file"], encoding="utf-8") as f:
    first = json.loads(f.readline())
print("Generated admission doc roles:", [d.get("doc_type_cn") for d in first["docs"]])
print(first["docs"][0]["text"][:400])

# 用完及时停服释放显存
cb.stop_vllm(srv)